# Exploratory Data Analysis: Vehicle Advertisements

This notebook explores the `vehicles_us.csv` dataset. The goal is to inspect the data, check for duplicates and missing values, restore missing values where reasonable, and create visualizations that help explain vehicle price patterns.

## 1. Import Libraries and Load Data

We start by importing pandas and Plotly Express, then reading the vehicle advertisement dataset.

In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('../vehicles_us.csv')
df.head()

## 2. Initial Data Inspection

Next, we check the structure of the dataset, including column names, data types, and non-null counts.

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Duplicate Check

Checking for duplicated rows is important because duplicates can distort summary statistics and visualizations.

In [ ]:
duplicate_count = df.duplicated().sum()
print('Number of duplicated rows:', duplicate_count)

## 4. Missing Value Check

We check missing values before cleaning. Some missing values can be restored using assumptions based on the meaning of the columns.

In [ ]:
df.isna().sum()

## 5. Missing Value Restoration

Assumptions used:

- `is_4wd`: The column contains `1` for 4WD vehicles, so missing values are assumed to represent non-4WD vehicles and are filled with `0`.
- `paint_color`: Missing colors are replaced with `'Unknown'` because the true color cannot be inferred reliably.
- `model_year`: Missing values are filled with the median model year for the same vehicle model.
- `cylinders`: Missing values are filled with the median cylinder count for the same model.
- `odometer`: Missing values are first filled using median odometer values for the same model and model year, then by model, and finally by the overall median if needed.

In [ ]:
df_clean = df.copy()

df_clean['is_4wd'] = df_clean['is_4wd'].fillna(0)
df_clean['paint_color'] = df_clean['paint_color'].fillna('Unknown')

df_clean['model_year'] = df_clean['model_year'].fillna(
    df_clean.groupby('model')['model_year'].transform('median')
)
df_clean['model_year'] = df_clean['model_year'].fillna(df_clean['model_year'].median())

df_clean['cylinders'] = df_clean['cylinders'].fillna(
    df_clean.groupby('model')['cylinders'].transform('median')
)
df_clean['cylinders'] = df_clean['cylinders'].fillna(df_clean['cylinders'].median())

df_clean['odometer'] = df_clean['odometer'].fillna(
    df_clean.groupby(['model', 'model_year'])['odometer'].transform('median')
)
df_clean['odometer'] = df_clean['odometer'].fillna(
    df_clean.groupby('model')['odometer'].transform('median')
)
df_clean['odometer'] = df_clean['odometer'].fillna(df_clean['odometer'].median())

df_clean.isna().sum()

## 6. Price Distribution

The first visualization shows the distribution of vehicle prices. Very expensive vehicles can act as outliers, so this chart excludes vehicles above $100,000 to make the main distribution easier to read.

In [ ]:
price_filtered = df_clean[df_clean['price'] <= 100000]

fig = px.histogram(
    price_filtered,
    x='price',
    nbins=50,
    title='Distribution of Vehicle Prices Under $100,000'
)
fig.show()

### Price Distribution Findings

Most vehicles are concentrated in the lower price range. The distribution is right-skewed, meaning there are fewer high-priced vehicles and many more lower-priced used vehicles.

## 7. Price vs. Odometer

This scatter plot compares vehicle price with odometer readings. It helps show whether vehicles with higher mileage tend to be cheaper.

In [ ]:
fig = px.scatter(
    price_filtered,
    x='odometer',
    y='price',
    title='Vehicle Price vs. Odometer',
    opacity=0.5
)
fig.show()

### Price vs. Odometer Findings

The scatter plot suggests that vehicles with higher odometer readings often have lower prices, although the relationship is not perfectly linear. Other factors, such as model, age, condition, and vehicle type, also likely affect price.

## 8. Summary

This exploratory analysis found that the dataset contains missing values in several important columns. Instead of dropping rows, missing values were restored using reasonable assumptions and contextual medians. The visualizations show that vehicle prices are strongly right-skewed, with most vehicles listed at lower prices and a smaller number of expensive outliers. The scatter plot also suggests that higher mileage is generally associated with lower price, although many other vehicle characteristics influence price as well.